# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR\u2072 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields with reference to their `@id` fields.

In [ ]:
# List available record sets and their fields, referencing by @id
record_set_ids = []
for rs in dataset.metadata.record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.identifier}")
    record_set_ids.append(rs.identifier)
    print("  Fields and Columns:")
    for field in rs.fields:
        field_id = field.identifier
        print(f"    Field: {field.name} (@id: {field_id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    print('-'*50)

# For demo, print the first 2 records of the first record set
if record_set_ids:
    print(f"\nFirst 2 records of RecordSet {record_set_ids[0]}:")
    for idx, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        if idx>=2:
            break
        print(rec)
else:
    print("No record sets found.")

## 3. Data Extraction
Load one or more record sets into pandas DataFrames for analysis.

Reference is always by `@id`.

In [ ]:
# Extract records from all record sets into DataFrames, referenced by @id
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded DataFrame for RecordSet @id='{rsid}' with {len(df)} rows and columns: {df.columns.tolist()}")

# Examine columns for the main record set (choose the first)
focus_record_set_id = record_set_ids[0] if record_set_ids else None
if focus_record_set_id:
    print(f"\nField/column names in DataFrame for main record set (@id='{focus_record_set_id}'):")
    print(dataframes[focus_record_set_id].columns.tolist())
    dataframes[focus_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping records.

In [ ]:
# Select a numeric field for analysis, referencing by its @id
# Below, we look for commonly named numeric columns
df = dataframes[focus_record_set_id]
numeric_field_candidates = [
    c for c in df.columns if 'coverage' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'abundance' in c.lower() or pd.api.types.is_numeric_dtype(df[c])
]
if len(numeric_field_candidates) == 0:
    raise Exception('No numeric field found for this record set. Please inspect your dataset.')
numeric_field_id = numeric_field_candidates[0]
# For grouping, try to find a string/binary column
non_numeric_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
group_field_id = non_numeric_candidates[0] if non_numeric_candidates else None

threshold = df[numeric_field_id].quantile(0.75)  # For example, filter on top quartile
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the selected numeric field
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nAfter normalization of '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, normalized_col]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped mean statistics by '{group_field_id}':")
    print(grouped_df[[numeric_field_id, normalized_col]].head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized version.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of '{numeric_field_id}' in RecordSet @id={focus_record_set_id}")
plt.xlabel(numeric_field_id)
plt.show()

plt.figure(figsize=(12,5))
sns.histplot(filtered_df[normalized_col].dropna(), bins=30, kde=True, color='orange')
plt.title(f"Distribution of normalized '{numeric_field_id}' (filtered, >Q3)")
plt.xlabel(normalized_col)
plt.show()

## 6. Conclusion
This notebook has demonstrated basic loading, exploration, filtering, normalization, grouping, and visualization operations using the FAIR\u2072 Croissant dataset on extracellular vesicle proteins.

You can use the above code template to experiment with other record sets, fields, and custom analyses relevant to your research.